# Dataset Exploration & Sanity Check

Quick notebook to verify the downloaded dataset looks correct.

In [ ]:
from pathlib import Path
import yaml
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
def find_data_yaml():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base / "data" / "data.yaml", base / "data.yaml"):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError("Could not find data/data.yaml")

DATA_YAML = find_data_yaml()
DATA_ROOT = DATA_YAML.parent

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print("Data YAML:", DATA_YAML)
print("Classes:", data_cfg.get("names", []))
print("Num classes:", data_cfg.get("nc", "?"))


In [ ]:
def split_dir(split):
    path = Path(data_cfg[split])
    return path if path.is_absolute() else (DATA_ROOT / path).resolve()

for split in ["train", "val", "test"]:
    img_dir = split_dir(split)
    if img_dir.exists():
        count = len(list(img_dir.iterdir()))
        print(f"{split}: {count} images")
    else:
        print(f"{split}: not found at {img_dir}")


In [ ]:
train_imgs = sorted(split_dir("train").glob("*"))

if train_imgs:
    fig, axes = plt.subplots(1, min(4, len(train_imgs)), figsize=(16, 4))
    axes_list = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax, img_path in zip(axes_list, train_imgs[:4]):
        ax.imshow(Image.open(img_path))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No training images found.")
